# 🎯 Module 13 · 02 — Text Preprocessing

> **Stage:** Machine Learning → NLP  
> **Level:** Beginner → Intermediate  
> **Module:** 13 — NLP Basics / 02 — Text Preprocessing  
> **Date:** 26 January 2026  
> **Previous:** 01 — NLP Fundamentals  
> **Next:** 03 — Text Representation

---

Real text is messy. **Text preprocessing** cleans and prepares raw text so it can be turned into numerical features for ML models.

In this notebook we'll cover:

- Why preprocess text
- Lowercasing
- HTML removal
- URL handling
- Punctuation
- Whitespace normalization
- Numbers
- Emojis & special characters
- Tokenization
- Stop words
- Stemming vs Lemmatization
- Normalization
- Spelling errors & Levenshtein distance
- A full practical pipeline

## 📚 Learning Objectives

By the end of this notebook you should be able to:

- Explain the goal of text preprocessing
- Apply lowercasing, HTML/URL removal, punctuation handling
- Normalize whitespace and numbers
- Handle emojis and special characters appropriately
- Tokenize text at word / sentence / character level
- Decide when to remove stop words
- Apply stemming and lemmatization
- Handle common spelling errors
- Compute **Levenshtein distance**
- Build a task-aware preprocessing pipeline

---
## ⚙️ 0. Setup & Imports

In [ ]:
import re
import string
from collections import Counter
import matplotlib.pyplot as plt

print('✅ Setup ready.')

In [ ]:
# A purposely messy raw text sample to reuse throughout the notebook
raw_text = '''
<p>OMG!!! This PRODUCT is AMAZING 😍😍</p>
Visit https://example.com NOW!!!  I bought 3 phones for $500.
'''

print('Raw text:')
print(repr(raw_text))
print()
print('Length (chars):', len(raw_text))

---
## 🧠 1. Why Preprocess Text?

Real-world text is **messy**:

```text
"OMG!!! This PRODUCT is AMAZING 😍😍
Visit https://example.com NOW!!!"
```

It contains:

- Uppercase letters
- Repeated punctuation
- Emojis
- URLs
- Extra whitespace
- Mixed formatting

A preprocessing pipeline can turn this into something more consistent:

```text
"this product is amazing"
```

> ⚠️ The exact preprocessing depends on the **NLP problem**. There is no single correct pipeline.

---
## 🔠 2. Lowercasing

```python
text = "I LOVE Python"
text.lower()
# → "i love python"
```

### Why?

Without lowercasing, these become different tokens:

```text
Python
python
PYTHON
```

### ⚠️ When NOT to lowercase

```text
US    → country
us    → pronoun
```

Lowercase **only when it makes sense for the task**.

In [ ]:
# Lowercasing demo
text = "I LOVE Python"
print('Original :', text)
print('Lowercase:', text.lower())

---
## 🌐 3. Removing HTML

Web data often contains HTML tags:

```html
<p>This product is amazing</p>
```

We usually want just:

```text
This product is amazing
```

Libraries like **BeautifulSoup** handle this. But a **regex** also works for simple cases.

In [ ]:
# Remove HTML tags with a simple regex
html_text = "<p>This product is amazing</p>"

cleaned = re.sub(r'<[^>]+>', '', html_text)

print('Original:', html_text)
print('Cleaned :', cleaned)

# Optional: use BeautifulSoup if installed
try:
    from bs4 import BeautifulSoup
    bs_clean = BeautifulSoup(html_text, 'html.parser').get_text()
    print('bs4     :', bs_clean)
except ImportError:
    print('bs4 not installed - pip install beautifulsoup4 to enable.')

---
## 🔗 4. Handling URLs

URLs often carry little text-classification signal, but they can be **very meaningful** for spam / phishing detection.

### Options

- **Remove** them entirely
- Replace with a **special token** like `<URL>`

Replacing is often the better default — it keeps the *fact* that a URL existed without leaking the actual URL content.

In [ ]:
url_text = "Visit https://example.com for more information"

# Option A: remove entirely
removed = re.sub(r'http\S+|www\.\S+', '', url_text)
removed = ' '.join(removed.split())   # clean extra spaces

# Option B: replace with a token
replaced = re.sub(r'http\S+|www\.\S+', '<URL>', url_text)

print('Original         :', url_text)
print('Remove URL       :', removed)
print('Replace with tag :', replaced)

---
## ❗ 5. Punctuation

Punctuation includes:

```text
. , ! ? ; : " ' ( ) -
```

### Removing punctuation

```text
"Hello, world!" → "Hello world"
```

### ⚠️ But punctuation can carry meaning

```text
"Great!"     → excitement
"Great..."   → hesitation
"Great?"     → uncertainty
```

> Punctuation removal is **task-dependent**.

In [ ]:
text = "Hello, world!"

# Translate table removes all ASCII punctuation
no_punct = text.translate(str.maketrans('', '', string.punctuation))

print('Original       :', text)
print('No punctuation :', no_punct)

# Alternative: keep punctuation that matters
keep = "!?"
keep_set = set(string.punctuation) - set(keep)
partial = text.translate(str.maketrans('', '', ''.join(keep_set)))
print(f'Keep {keep!r}     : {partial}')

---
## ␣ 6. Whitespace Cleaning

Raw text can contain extra spaces, tabs, and newlines:

```text
"Hello     world\n\nPython"
```

We can normalize it:

```python
text = " ".join(text.split())
```

Result:

```text
"Hello world Python"
```

In [ ]:
messy = "Hello     world\n\nPython\t\t3"
clean = ' '.join(messy.split())

print('Messy :', repr(messy))
print('Clean :', repr(clean))

---
## 🔢 7. Numbers

Numbers can be:

- **Removed** (for general text classification)
- **Kept** (for financial, medical, scientific text)
- **Replaced** with a special token

```text
"I bought 3 phones for $500"
```

> ⚠️ **Never remove numbers automatically without considering the task.**

In [ ]:
num_text = "I bought 3 phones for $500"

# Remove
no_nums = re.sub(r'\d+', '', num_text)
no_nums = ' '.join(no_nums.split())

# Replace with a token
token_nums = re.sub(r'\d+', '<NUM>', num_text)

print('Original     :', num_text)
print('Remove nums  :', no_nums)
print('Replace nums :', token_nums)

---
## 😍 8. Emojis & Special Characters

Emojis carry **sentiment**:

```text
😍 → positive signal
😡 → negative signal
```

Depending on the task, emojis can be:

- **Kept** as-is
- **Removed**
- **Converted to text** (e.g., 😍 → "heart_eyes")
- **Replaced** with a special token

> Removing emojis blindly can hurt sentiment models.

In [ ]:
# Detect emojis with a simple Unicode-range check
def find_emojis(text):
    return [c for c in text if ord(c) > 0xFFFF]

emoji_text = "This is amazing 😍🔥"
print('Text      :', emoji_text)
print('Emojis    :', find_emojis(emoji_text))

# Optional: use the `emoji` library for names
try:
    import emoji
    print('Names     :', [emoji.demojize(c) for c in find_emojis(emoji_text)])
except ImportError:
    print('emoji lib not installed - pip install emoji to enable name conversion.')

---
## ✂️ 9. Tokenization

**Tokenization** breaks text into smaller units called **tokens**.

```text
"I love Python" → ["I", "love", "Python"]
```

### Tokenization levels

| Type | Example |
| --- | --- |
| Word | `["I", "love", "Python"]` |
| Sentence | `["I love Python."]` |
| Character | `["P", "y", "t", ...]` |
| Subword | Pieces of words |

In [ ]:
text = "I love Python. It is powerful."

# Word tokenization (simple)
words = text.split()

# Better word tokenization with regex
words_re = re.findall(r"\b\w+\b", text)

# Sentence tokenization (simple regex on .!?)
sentences = re.split(r'(?<=[.!?])\s+', text)

# Character tokenization (just the first 10 chars)
chars = list(text[:10])

print('Text          :', text)
print('Words (split) :', words)
print('Words (regex) :', words_re)
print('Sentences     :', sentences)
print('Chars (10)    :', chars)

---
## 🚫 10. Stop Words

**Stop words** are very common words with limited information for some tasks:

```text
the, is, a, an, of, to, in
```

### Example

```text
"the laptop is good" → "laptop good"
```

### ⚠️ When NOT to remove them

```text
"I do not like this."
```

Removing **not** changes the meaning completely:

```text
"I like this."
```

> Stop-word removal is **optional and task-dependent**.

In [ ]:
# A small custom stopword set (avoids NLTK download requirement)
STOP_WORDS = {
    'the', 'is', 'a', 'an', 'of', 'to', 'in', 'and', 'or', 'on', 'for', 'this', 'that'
}

text = "The laptop is a good option for this task"
tokens = text.lower().split()
filtered = [t for t in tokens if t not in STOP_WORDS]

print('Original :', tokens)
print('Filtered :', filtered)

# The danger: negation
neg = "I do not like this product"
neg_tokens = neg.lower().split()
neg_filtered = [t for t in neg_tokens if t not in STOP_WORDS]
print()
print('With negation :', neg_tokens)
print('After removal :', neg_filtered, '  <- meaning destroyed!')

---
## 🌱 11. Stemming

**Stemming** reduces words to a **root-like form** using rules.

```text
playing / played / plays  →  play
```

Common algorithm: **Porter Stemmer** (NLTK).

### Pros / Cons

- ✅ Fast, simple
- ❌ Can produce non-words (`studies → studi`)

In [ ]:
# Try NLTK Porter Stemmer, fall back to a tiny rule-based stemmer if not installed
try:
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
    def stem(word):
        return stemmer.stem(word)
except ImportError:
    print('NLTK not installed - using a tiny rule-based fallback.')
    def stem(word):
        for suf in ('ing', 'ed', 'es', 's'):
            if word.endswith(suf) and len(word) > len(suf) + 2:
                return word[:-len(suf)]
        return word

for w in ['playing', 'played', 'plays', 'studies', 'running']:
    print(f'{w:<10} → {stem(w)}')

---
## 📖 12. Lemmatization

**Lemmatization** reduces a word to its **proper dictionary base form**.

```text
better  → good
running → run
studies → study
```

### Stemming vs Lemmatization

| Stemming | Lemmatization |
| --- | --- |
| Rule-based shortening | Linguistic normalization |
| Faster | Slower |
| May produce non-words | Produces valid words |
| Less linguistically informed | More linguistically informed |

In [ ]:
# Try spaCy lemmatizer, fall back to a small manual map
try:
    import spacy
    nlp = spacy.load('en_core_web_sm')
    def lemmatize(word):
        return nlp(word)[0].lemma_
except (ImportError, OSError):
    print('spaCy / en_core_web_sm not available - using a small manual map.')
    LEMMA_MAP = {
        'better': 'good', 'best': 'good',
        'running': 'run', 'ran': 'run',
        'studies': 'study', 'studied': 'study',
        'cars': 'car', 'children': 'child',
    }
    def lemmatize(word):
        return LEMMA_MAP.get(word.lower(), word.lower())

for w in ['better', 'running', 'studies', 'cars', 'children']:
    print(f'{w:<10} → {lemmatize(w)}')

---
## 🔄 13. Normalization

Text normalization makes different forms more consistent.

Examples:

```text
"HELLO"            → "hello"
"can't"            → "cannot"
"  hello   world  " → "hello world"
```

Common normalization:

- Lowercasing
- Whitespace normalization
- Contraction handling
- Spelling normalization
- Unicode normalization

In [ ]:
# Contraction handling (a small common map)
CONTRACTIONS = {
    "can't": "cannot",
    "won't": "will not",
    "don't": "do not",
    "isn't": "is not",
    "i'm":   "i am",
    "it's":  "it is",
}

def expand_contractions(text):
    for k, v in CONTRACTIONS.items():
        text = re.sub(r'\b' + re.escape(k) + r'\b', v, text, flags=re.IGNORECASE)
    return text

samples = ["I can't believe it's here", "Don't worry, it's fine"]
for s in samples:
    print(f'{s:<30} → {expand_contractions(s)}')

---
## ✏️ 14. Spelling Errors

User-generated text often contains misspellings:

```text
recieve
definately
amazng
```

### Possible approaches

- Dictionary correction
- Edit distance
- Spell-checking libraries
- Domain-specific correction

> ⚠️ Automatic correction can change intended meaning — use carefully.

---
## 📏 15. Levenshtein Distance

**Levenshtein distance** = minimum number of single-character edits to transform one string into another.

Operations:

```text
Insertion
Deletion
Substitution
```

Example:

```text
cat → cut
```

Only **one** substitution is required:

```text
distance = 1
```

Used in:

- Spell correction
- Fuzzy matching
- Duplicate detection
- Search systems

In [ ]:
# Levenshtein distance — dynamic programming, no external library
def levenshtein(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m + 1): dp[i][0] = i
    for j in range(n + 1): dp[0][j] = j
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            cost = 0 if a[i - 1] == b[j - 1] else 1
            dp[i][j] = min(
                dp[i - 1][j] + 1,        # deletion
                dp[i][j - 1] + 1,        # insertion
                dp[i - 1][j - 1] + cost  # substitution
            )
    return dp[m][n]

pairs = [('cat', 'cut'), ('kitten', 'sitting'), ('python', 'pyhton'), ('hi', 'hi')]
for a, b in pairs:
    print(f'{a!r:>10} vs {b!r:<10} → distance = {levenshtein(a, b)}')

---
## 🏗️ 16. A Practical Preprocessing Pipeline

```text
Raw Text
   ↓
Remove HTML
   ↓
Handle URLs
   ↓
Normalize whitespace
   ↓
Lowercase
   ↓
Tokenize
   ↓
Optional stop-word removal
   ↓
Optional stemming / lemmatization
   ↓
Clean Tokens
   ↓
TF-IDF / BoW
```

### Not every step is for every project

| Task | Special consideration |
| --- | --- |
| Sentiment Analysis | Keep negation and emojis |
| Spam Detection | URLs may be useful features |
| Search | Stemming can help match word forms |

In [ ]:
# A reusable, task-aware preprocessing pipeline
def preprocess(text,
               remove_html=True,
               replace_urls=True,
               lowercase=True,
               remove_punct=False,
               remove_numbers=False,
               remove_stopwords=False,
               stem_tokens=False,
               stop_words=None):
    if remove_html:
        text = re.sub(r'<[^>]+>', ' ', text)

    if replace_urls:
        text = re.sub(r'http\S+|www\.\S+', '<URL>', text)

    text = ' '.join(text.split())  # whitespace

    if lowercase:
        text = text.lower()

    if remove_punct:
        text = text.translate(str.maketrans('', '', string.punctuation))

    if remove_numbers:
        text = re.sub(r'\d+', '', text)

    tokens = re.findall(r"\b\w+\b", text) if not remove_punct else text.split()

    if remove_stopwords and stop_words:
        tokens = [t for t in tokens if t not in stop_words]

    if stem_tokens:
        tokens = [stem(t) for t in tokens]

    return tokens

stop = {'the', 'is', 'a', 'an', 'of', 'to', 'in', 'and', 'for'}

print('=== Sentiment-oriented (keeps negation, keeps emojis, keeps punctuation) ===')
print(preprocess("<p>I don't love this 😍</p>", remove_punct=False))

print()
print('=== Bag-of-words-oriented (aggressive cleaning) ===')
print(preprocess(raw_text, remove_punct=True, remove_numbers=True, remove_stopwords=True, stop_words=stop))

---
## 🏋️ 17. Hands-On Exercises

### Exercise 1 — End-to-end cleaning
Write a function that performs in order:
1. Remove HTML
2. Replace URLs with `<URL>`
3. Lowercase
4. Normalize whitespace

Test it on:
```python
text = '<div>NEWS!! Visit https://news.com for INFO</div>'
```

### Exercise 2 — Punctuation trade-off
Compare tokenization of:
```text
"I love it!!!"
"I love it..."
"I love it?"
```
With and without punctuation removal. What information is lost?

### Exercise 3 — Stemming vs Lemmatization
Apply both to:
```text
['studies', 'studied', 'better', 'running', 'cars']
```
Compare the outputs.

### Exercise 4 — Levenshtein
Compute Levenshtein distance for:
- `('kitten', 'sitting')`
- `('hello', 'hallo')`
- `('run', 'running')`

### Exercise 5 — Task-aware pipeline
Design a preprocessing pipeline for **spam detection**:
- What would you keep?
- What would you remove?
- Why?

---
## 🎤 18. Interview Questions

1. **Why preprocess text?**  
   To remove noise while preserving task-relevant information before vectorization.

2. **When should you NOT lowercase?**  
   When case carries meaning (`US` vs `us`, product names, acronyms).

3. **How do you remove HTML?**  
   BeautifulSoup, or a simple regex for basic cases.

4. **Should you remove URLs?**  
   It depends — replace with a token when URL presence matters (spam), otherwise remove.

5. **Should you always remove punctuation?**  
   No — punctuation can carry sentiment and intent (`!`, `?`, `...`).

6. **When should numbers be kept?**  
   For financial, medical, product, or numeric-heavy domains.

7. **When should stop words be kept?**  
   When negation matters (sentiment) or when common words carry semantic value.

8. **Stemming vs Lemmatization?**  
   Stemming is faster and rule-based; lemmatization is linguistically informed and returns valid words.

9. **What is Levenshtein distance?**  
   Minimum single-character edits required to transform one string into another.

10. **What is the biggest preprocessing mistake?**  
    Removing information the model needed (e.g., stripping negation for sentiment).

11. **Why should preprocessing be task-aware?**  
    Different tasks require different signals — sentiment ≠ spam ≠ search.

12. **How does preprocessing fit into a Pipeline?**  
    The preprocessing steps should be part of the fitted Pipeline so they apply identically during train and inference.

---
## 🏁 19. Key Takeaways

```text
Raw Text
   ↓
Clean
   ↓
Normalize
   ↓
Tokenize
   ↓
Optional linguistic processing
   ↓
Ready for representation
```

### 🧭 Golden Rule

> **Good preprocessing removes noise WITHOUT removing useful information.**

### You now understand

- Lowercasing
- HTML removal
- URL handling
- Punctuation
- Whitespace
- Numbers
- Emojis
- Tokenization
- Stop words
- Stemming
- Lemmatization
- Normalization
- Spelling errors
- Levenshtein distance
- Task-dependent preprocessing

---

### 🔗 What's Next?

**03 — Text Representation**

We convert processed text into **numerical representations** that machine learning models can use — starting with Bag of Words and TF-IDF.